# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [3]:
# load necessary packages
# copied from lecture 9 

import re
import Bio.SeqIO
import Bio.Seq

In [4]:
#get seqreads
seqreads = list(Bio.SeqIO.parse('barcodes_R1.fastq', 'fastq'))

In [20]:
def get_strain_barcodes(seqreads, bclen = 16, upstream = 'GCGGCCGCCT'):
    """Identify barcode with known upstream sequence and its strain (HA/NA).
    
    Parameters
    ----------
    seqreads : Seq object list
        Nucleotide sequence read matching UPSTREAM-BARCODE in reverse orientation.
        IN LIST FORMATION (all reads you want checked.)
    bclen : int
        Length of barcode
    upstream: str
        Sequence upstream of the barcode.
        
    Output
    -------
        prints different values wanted such as number of reads per strain or
        barcode with most reads, etc.

    Example
    -------
    >>> read_barcode(Bio.Seq.Seq('TTTTTTTTTTTTTTTTGCGGCCGCCT-"EndHA"'), bclen=16)
    'AAAAAAAAAAAAAAAA' - in the HA dictionary
        
    """

    #Defines the NA/HA patterns
    NA_END = re.compile('ATGGTGCACTATTATTTATCTATCGTG')
    HA_END = re.compile('ATGGTGCATCATTATATGCAAATCCGG')

    #Initializes variables of interest
    COUNTS_NA = {}
    COUNTS_HA = {}

    number_NA_reads = 0
    number_HA_reads = 0

    invalid_count_NA = 0
    invalid_count_HA = 0

    #cycles throuh sequences, checking first for if it is an HA or NA strain
    # then, if it is a valid strain, adding the barcode to the dictionary
    for seq in seqreads:
        
        seq.upper()
        seq_str = str(seq)

        #checks strain
        matchNA = NA_END.search(seq_str)
        matchHA = HA_END.search(seq_str)

        # adds barcode to NA dictionary if it exists
        if matchNA:

            number_NA_reads += 1
            pat = re.compile('(?P<barcode>[A-Z]{'+str(bclen)+'})'+upstream)

            match = pat.search(seq_str)

            if match is None:
                invalid_count_NA += 1
            else:
                barcode = match.group('barcode')

                if barcode in COUNTS_NA:
                    COUNTS_NA[barcode] += 1
                else:
                    COUNTS_NA[barcode] = 1  
                        
        # adds barcode to HA dictionary if it exists
        elif matchHA:
            
            number_HA_reads += 1
            pat = re.compile('(?P<barcode>[A-Z]{'+str(bclen)+'})'+upstream)

            match = pat.search(seq_str)

            if match is None:
                invalid_count_HA += 1
            else:
                barcode = match.group('barcode')

                if barcode in COUNTS_HA:
                    COUNTS_HA[barcode] += 1
                else:
                    COUNTS_HA[barcode] = 1  
    

    # prints the total number of reads per strain
    print(f"total of NA reads: {number_NA_reads}")
    print(f"total of NA reads: {number_HA_reads}")

    # prints number of invalid barcodes
    print(f"number of invalid NA barcodes: {invalid_count_NA}")
    print(f"number of invlaid HA barcodes: {invalid_count_HA}") 

    # finds that max value's key in the given dictionary
    max_NA = max(COUNTS_NA, key = COUNTS_NA.get)
    max_HA = max(COUNTS_HA, key = COUNTS_HA.get)

    # prints the max count key and counts
    print(f'NA barcode with most counts is \'{max_NA}\' with {COUNTS_NA[max_NA]} counts')
    print(f'HA barcode with most counts is: \'{max_HA}\' with {COUNTS_HA[max_HA]} counts')

get_strain_barcodes(seqreads)

total of NA reads: 4120
total of NA reads: 5406
number of invalid NA barcodes: 210
number of invlaid HA barcodes: 159
NA barcode with most counts is 'CCCGGGGAGAACTGGT' with 152 counts
HA barcode with most counts is: 'TTAATGTCGGGTCGGG' with 155 counts
